SDG + INEGI — Exploración de indicadores
Notebook para obtener indicadores del Banco Mundial (ODS) e INEGI,
y graficarlos superpuestos para un país o región.

**Instalación:**
```
pip install wbgapi requests pandas matplotlib
```

In [ ]:

import json
import time
from pathlib import Path
from dataclasses import dataclass, field

import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import wbgapi as wb

In [ ]:
#── Configuración global ────────────────────────────────────────────────────

CACHE_DIR = Path("data_cache")
CACHE_DIR.mkdir(exist_ok=True)

# País por defecto (ISO-3)
DEFAULT_COUNTRY = "MEX"

# Rango de años
YEAR_START = 2000
YEAR_END   = 2023

# Paleta de colores fija por fuente
COLORS = {
    "worldbank": "#1D9E75",   # teal — ODS / Banco Mundial
    "inegi":     "#534AB7",   # purple — INEGI
    "alt1":      "#D85A30",   # coral
    "alt2":      "#BA7517",   # amber
}

In [ ]:
#── Dataclass para un indicador ────────────────────────────────────────────

@dataclass
class Indicator:
    """Representa un indicador de cualquier fuente con sus metadatos."""
    code: str                        # código en la API origen
    label: str                       # nombre corto para gráficas
    source: str                      # "worldbank" | "inegi"
    unit: str = ""                   # ej. "%" | "personas" | "índice"
    country: str = DEFAULT_COUNTRY
    extra_params: dict = field(default_factory=dict)  # params adicionales

In [ ]:
# %% ── Catálogo de indicadores ODS ────────────────────────────────────────────
# Fuente primaria: World Bank API (wbgapi)
# Códigos verificados contra: https://data.worldbank.org/indicator
#
# Notas de cobertura:
#   - 9.1.1 y 13.3.1 no tienen código WB exacto al nivel del indicador ODS —
#     se usa el proxy más cercano disponible. Se documenta la diferencia.
#   - 5.5.2 tiene cobertura irregular (años sueltos, no serie continua).
#   - Para datos por sector (GHG) y municipales, se necesitará INEGI/SEMARNAT.

CATALOG: dict[str, Indicator] = {

    # ── ODS 8.6.1 ─────────────────────────────────────────────────────────────
    # Proporción de jóvenes (15-24) que no estudian, no trabajan ni se capacitan
    "neet": Indicator(
        code="SL.UEM.NEET.ZS",
        label="Jóvenes NEET (8.6.1)",
        source="worldbank",
        unit="% jóvenes 15-24",
        country="MEX",
    ),

    # ── ODS 9.1.1 ─────────────────────────────────────────────────────────────
    # Proxy: % población rural con acceso a carretera transitable todo el año
    # Nota: el indicador exacto (Rural Access Index) solo está en datos de proyecto
    # del Banco Mundial, no en su API pública. Este es el proxy más cercano.
    "acceso_carretera": Indicator(
        code="IS.ROD.PAVE.ZS",
        label="Carreteras pavimentadas (proxy 9.1.1)",
        source="worldbank",
        unit="% del total de carreteras",
        country="MEX",
    ),

    # ── ODS 3.4.2a ────────────────────────────────────────────────────────────
    # Tasa de mortalidad por suicidio (por 100,000 habitantes)
    "suicidio": Indicator(
        code="SH.STA.SUIC.P5",
        label="Mortalidad por suicidio (3.4.2)",
        source="worldbank",
        unit="por 100,000 hab.",
        country="MEX",
    ),

    # ── ODS 5.5.2 ─────────────────────────────────────────────────────────────
    # Proporción de mujeres en cargos directivos y gerenciales
    # Advertencia: cobertura irregular para México — años con datos: ~2000, 2010, 2019
    "mujeres_directivas": Indicator(
        code="SL.EMP.SMGT.FE.ZS",
        label="Mujeres en puestos directivos (5.5.2)",
        source="worldbank",
        unit="% del total de directivos",
        country="MEX",
    ),

    # ── ODS 13.3.1 ────────────────────────────────────────────────────────────
    # Proxy: emisiones totales de GHG (kt CO2 equivalente)
    # Nota: el indicador oficial pide desglose por sector (energía, AFOLU, residuos).
    # El WB solo publica el total. Para desglose sectorial usar SEMARNAT o Climate Watch.
    "ghg_total": Indicator(
        code="EN.ATM.GHGT.KT.CE",
        label="Emisiones GHG totales (proxy 13.3.1)",
        source="worldbank",
        unit="kt CO₂ equivalente",
        country="MEX",
    ),

    # ── ODS 10.2.1a ───────────────────────────────────────────────────────────
    # Proxy: tasa de crecimiento del ingreso del 40% más pobre
    # Nota: el indicador exacto (% bajo la mediana) está en microdatos ENIGH/CEPAL.
    # Este es el proxy disponible en WB para desigualdad de ingresos en México.
    "ingreso_40_inferior": Indicator(
        code="SI.SPR.PC40.ZG",
        label="Crecimiento ingreso 40% inferior (proxy 10.2.1)",
        source="worldbank",
        unit="% anual",
        country="MEX",
    ),
}

In [ ]:
#── Catálogo de indicadores ─────────────────────────────────────────────────
# Edita este dict para agregar los indicadores que te interesen.
# World Bank codes: https://data.worldbank.org/indicator
# INEGI series:     https://www.inegi.org.mx/servicios/api_indicadores.html

CATALOG: dict[str, Indicator] = {

    # ── ODS 1 — Fin de la pobreza ──────────────────────────────────────────────
    "pobreza_extrema": Indicator(
        code="SI.POV.DDAY",
        label="Pobreza extrema (<$2.15/día)",
        source="worldbank",
        unit="%",
    ),
    "gini": Indicator(
        code="SI.POV.GINI",
        label="Coeficiente Gini",
        source="worldbank",
        unit="índice",
    ),

    # ── ODS 3 — Salud ─────────────────────────────────────────────────────────
    "mortalidad_infantil": Indicator(
        code="SH.DYN.MORT",
        label="Mortalidad infantil (por 1,000)",
        source="worldbank",
        unit="por 1,000 nacidos",
    ),
    "esperanza_vida": Indicator(
        code="SP.DYN.LE00.IN",
        label="Esperanza de vida al nacer",
        source="worldbank",
        unit="años",
    ),

    # ── ODS 4 — Educación ─────────────────────────────────────────────────────
    "matricula_primaria": Indicator(
        code="SE.PRM.ENRR",
        label="Matrícula primaria neta",
        source="worldbank",
        unit="%",
    ),

    # ── ODS 5 — Género ────────────────────────────────────────────────────────
    "brecha_laboral_genero": Indicator(
        code="SL.TLF.CACT.FE.ZS",
        label="Tasa actividad laboral femenina",
        source="worldbank",
        unit="%",
    ),

    # ── ODS 8 — Trabajo y crecimiento ─────────────────────────────────────────
    "desempleo": Indicator(
        code="SL.UEM.TOTL.ZS",
        label="Desempleo total",
        source="worldbank",
        unit="%",
    ),
    "pib_per_capita": Indicator(
        code="NY.GDP.PCAP.CD",
        label="PIB per cápita (USD)",
        source="worldbank",
        unit="USD",
    ),

    # ── INEGI — Indicadores nacionales ────────────────────────────────────────
    # BIE series IDs: https://www.inegi.org.mx/app/indicadores/
    # Estos IDs son ejemplos reales del Banco de Información Económica (BIE)
    "inegi_pib_trimestral": Indicator(
        code="493911",          # PIB a precios constantes, serie BIE
        label="PIB trimestral (INEGI)",
        source="inegi",
        unit="millones MXN",
    ),
    "inegi_desempleo": Indicator(
        code="444612",          # Tasa de desocupación nacional
        label="Desocupación nacional (INEGI-ENOE)",
        source="inegi",
        unit="%",
    ),
    "inegi_inflacion": Indicator(
        code="216064",          # INPC general
        label="Inflación (INPC)",
        source="inegi",
        unit="índice",
    ),
    
    
}

In [ ]:
CATALOG["neet"] = Indicator(
    code="SL.UEM.NEET.ZS",
    label="NEET total (8.6.1)",
    source="worldbank",
    unit="% jóvenes 15-24",
)
CATALOG["neet_f"] = Indicator(
    code="SL.UEM.NEET.FE.ZS",
    label="NEET mujeres (8.6.1)",
    source="worldbank",
    unit="% jóvenes 15-24",
)
CATALOG["neet_m"] = Indicator(
    code="SL.UEM.NEET.MA.ZS",
    label="NEET hombres (8.6.1)",
    source="worldbank",
    unit="% jóvenes 15-24",
)

In [ ]:
# ── ODS — fuente ONU SDG API ──────────────────────────────────────────────────
CATALOG["onu_neet"] = Indicator(
    code="8.6.1",
    label="Jóvenes NEET (8.6.1 ONU)",
    source="onu_sdg",
    unit="% jóvenes 15-24",
)
CATALOG["onu_mujeres_directivas"] = Indicator(
    code="5.5.2",
    label="Mujeres en dirección (5.5.2 ONU)",
    source="onu_sdg",
    unit="% puestos directivos",
)
CATALOG["onu_suicidio"] = Indicator(
    code="3.4.2",
    label="Mortalidad por suicidio (3.4.2 ONU)",
    source="onu_sdg",
    unit="por 100,000 hab.",
)
CATALOG["onu_ghg"] = Indicator(
    code="13.2.2",
    label="Emisiones GHG (13.2.2 ONU)",
    source="onu_sdg",
    unit="Mt CO₂ equivalente",
)

Añadimos fuentes de datos, aquí generamos el mètodo que habla a las API's específicas

Aqui fabricamos las llamadas a la api's y le damos el tratamiento a los datos para graficarlos

In [ ]:
# %% ── DataFactory completa ───────────────────────────────────────────────────
import json, time, requests
from pathlib import Path
import pandas as pd
import wbgapi as wb

ONU_SDG_BASE  = "https://unstats.un.org/sdgs/UNSDGAPIV5/v1"
ONU_AREA_CODE = "484"  # México ISO-3166 numérico

class DataFactory:

    WB_BASE    = "https://api.worldbank.org/v2"
    INEGI_BASE = "https://www.inegi.org.mx/app/indicadores/ws/series/BISE"

    def __init__(self, cache_dir: Path = CACHE_DIR, use_cache: bool = True):
        self.cache_dir = cache_dir
        self.use_cache = use_cache

    # ── Caché ─────────────────────────────────────────────────────────────────

    def _cache_path(self, key):
        return self.cache_dir / f"{key.replace('/', '_').replace('.', '_')}.json"

    def _load_cache(self, key):
        p = self._cache_path(key)
        if self.use_cache and p.exists():
            return json.loads(p.read_text())
        return None

    def _save_cache(self, key, data):
        self._cache_path(key).write_text(json.dumps(data, ensure_ascii=False))

    # ── World Bank ────────────────────────────────────────────────────────────

    def fetch_worldbank(self, indicator, year_start=YEAR_START, year_end=YEAR_END):
        cache_key = f"wb_{indicator.code}_{indicator.country}_{year_start}_{year_end}"
        cached = self._load_cache(cache_key)
        if cached:
            print(f"  [caché] {indicator.label}")
            records = cached
        else:
            print(f"  [fetch WB] {indicator.label}...")
            try:
                df_raw = wb.data.DataFrame(
                    indicator.code,
                    economy=indicator.country,
                    time=range(year_start, year_end + 1),
                )
                df_raw = df_raw.stack().reset_index()
                df_raw.columns = ["economy", "year", "value"]
                df_raw["year"] = df_raw["year"].astype(str).str.extract(r"(\d{4})").astype(int)
                records = df_raw[["year", "value"]].to_dict("records")
                self._save_cache(cache_key, records)
            except Exception as e:
                print(f"  Error: {e}")
                return pd.DataFrame(columns=["year", "value", "label", "unit", "source"])

        df = pd.DataFrame(records).dropna(subset=["value"]).sort_values("year").reset_index(drop=True)
        df["label"]  = indicator.label
        df["unit"]   = indicator.unit
        df["source"] = "worldbank"
        return df

    # ── INEGI ─────────────────────────────────────────────────────────────────

    def fetch_inegi(self, indicator, year_start=YEAR_START, year_end=YEAR_END):
        cache_key = f"inegi_{indicator.code}_{year_start}_{year_end}"
        cached = self._load_cache(cache_key)
        if cached:
            print(f"  [caché] {indicator.label}")
            records = cached
        else:
            print(f"  [fetch INEGI] {indicator.label}...")
            token = indicator.extra_params.get("token", "")
            if not token:
                print(f"  Sin token — agrega extra_params={{'token': 'TU_TOKEN'}}")
                return pd.DataFrame(columns=["year", "value", "label", "unit", "source"])
            url = f"{self.INEGI_BASE}/ObtenerDatosBIE?indicadores={indicator.code}&token={token}"
            try:
                resp = requests.get(url, timeout=10)
                resp.raise_for_status()
                raw = resp.json()
                records = []
                for obs in raw.get("Series", [{}])[0].get("OBSERVATIONS", []):
                    year = int(str(obs.get("TIME_PERIOD", "0"))[:4])
                    if year_start <= year <= year_end:
                        try:
                            records.append({"year": year, "value": float(obs["OBS_VALUE"])})
                        except (TypeError, ValueError):
                            pass
                self._save_cache(cache_key, records)
            except Exception as e:
                print(f"  Error: {e}")
                return pd.DataFrame(columns=["year", "value", "label", "unit", "source"])

        df = pd.DataFrame(records).dropna(subset=["value"]).sort_values("year").reset_index(drop=True)
        df["label"]  = indicator.label
        df["unit"]   = indicator.unit
        df["source"] = "inegi"
        return df

    # ── ONU SDG ───────────────────────────────────────────────────────────────

    def fetch_onu_sdg(self, indicator, year_start=YEAR_START, year_end=YEAR_END):
        cache_key = f"onu_{indicator.code.replace('.', '_')}_{ONU_AREA_CODE}_{year_start}_{year_end}"
        cached = self._load_cache(cache_key)
        if cached:
            print(f"  [caché] {indicator.label}")
            records = cached
        else:
            print(f"  [fetch ONU SDG] {indicator.label} ({indicator.code})...")
            url = (
                f"{ONU_SDG_BASE}/sdg/Indicator/Data"
                f"?indicator={indicator.code}"
                f"&areaCode={ONU_AREA_CODE}"
                f"&timePeriodStart={year_start}"
                f"&timePeriodEnd={year_end}"
            )
            try:
                resp = requests.get(url, timeout=15)
                resp.raise_for_status()
                raw = resp.json()
            except Exception as e:
                print(f"  Error: {e}")
                return pd.DataFrame(columns=["year", "value", "label", "unit", "source"])

            total = raw.get("totalElements", 0)
            if total == 0:
                print(f"  Sin datos para México en {indicator.code}")
                return pd.DataFrame(columns=["year", "value", "label", "unit", "source"])

            # dimensions es dict plano: {"Sex": "BOTHSEX", "Age": "15-24", ...}
            records = []
            series_vistas = set()
            for obs in raw.get("data", []):
                dims = obs.get("dimensions", {})

                sexo = dims.get("Sex", "BOTHSEX")
                area = dims.get("Location", "ALLAREA")

                if sexo not in ("BOTHSEX", "_T", "") or area not in ("ALLAREA", "_T", ""):
                    series_vistas.add(f"Sex={sexo}, Location={area}")
                    continue

                try:
                    year = int(obs["timePeriodStart"])
                    val  = float(obs["value"])
                    if year_start <= year <= year_end:
                        records.append({"year": year, "value": val})
                except (KeyError, ValueError, TypeError):
                    continue

            if not records:
                # Sin filtros — tomar todo lo que haya
                print(f"  Filtro Sex/Location no aplica, tomando todas las series...")
                for obs in raw.get("data", []):
                    try:
                        year = int(obs["timePeriodStart"])
                        val  = float(obs["value"])
                        if year_start <= year <= year_end:
                            records.append({"year": year, "value": val})
                    except (KeyError, ValueError, TypeError):
                        continue

            if not records:
                print(f"  Sin registros parseables para {indicator.code}")
                if series_vistas:
                    print(f"  Series vistas: {list(series_vistas)[:4]}")
                return pd.DataFrame(columns=["year", "value", "label", "unit", "source"])

            self._save_cache(cache_key, records)

        df = (pd.DataFrame(records)
                .dropna(subset=["value"])
                .drop_duplicates(subset=["year"])
                .sort_values("year")
                .reset_index(drop=True))
        df["label"]  = indicator.label
        df["unit"]   = indicator.unit
        df["source"] = "onu_sdg"
        print(f"  {len(df)} obs ({df['year'].min()}–{df['year'].max()})")
        return df

        #ODS blabla--------
        
    # ── Dispatcher ────────────────────────────────────────────────────────────

    def fetch(self, indicator, year_start=YEAR_START, year_end=YEAR_END):
        if indicator.source == "worldbank":
            return self.fetch_worldbank(indicator, year_start, year_end)
        elif indicator.source == "inegi":
            return self.fetch_inegi(indicator, year_start, year_end)
        elif indicator.source == "onu_sdg":
            return self.fetch_onu_sdg(indicator, year_start, year_end)
        else:
            raise ValueError(f"Fuente no soportada: {indicator.source}")

    def fetch_many(self, keys, year_start=YEAR_START, year_end=YEAR_END, delay=0.3):
        results = {}
        for key in keys:
            if key not in CATALOG:
                print(f"  '{key}' no en CATALOG")
                continue
            df = self.fetch(CATALOG[key], year_start, year_end)
            if not df.empty:
                results[key] = df
            time.sleep(delay)
        return results

    # ── CSV ───────────────────────────────────────────────────────────────────

    def to_csv(self, datasets, prefix=""):
        for key, df in datasets.items():
            path = self.cache_dir / f"{prefix}{key}.csv"
            df.to_csv(path, index=False)
            print(f"  guardado: {path}")

    def from_csv(self, keys, prefix=""):
        results = {}
        for key in keys:
            path = self.cache_dir / f"{prefix}{key}.csv"
            if not path.exists():
                print(f"  no encontrado: {path}")
                continue
            results[key] = pd.read_csv(path)
            print(f"  cargado: {path}")
        return results

Fin del "backend" a pintar!

I like it Picasso

In [ ]:
# %% ── Ejecución principal ────────────────────────────────────────────────────

factory = DataFactory(use_cache=True)

# ── Fuente: "live" o "csv" ────────────────────────────────────────────────────
FUENTE = "live"   # cambiar a "csv" para cargar sin hacer requests
FUENTE = "csv"

KEYS = [
    "gini",
    "neet",             # 8.6.1  — NEET jóvenes
    "onu_acceso_carretera", # 9.1.1  — lejos de carretera
    "onu_suicidio",         # 3.4.2  — mortalidad suicidio
    "onu_mujeres_directivas", # 5.5.2 — mujeres en dirección
    "onu_ghg",              # 13.2.2 — emisiones GHG
    "onu_ingreso_mediana",  # 10.2.1 — bajo mediana ingresos
       ]

if FUENTE == "csv":
    datos = factory.from_csv(KEYS)
else:
    datos = factory.fetch_many(KEYS, year_start=2005, year_end=2023)
    factory.to_csv(datos)   # guardar para la próxima vez

print("Disponibles:", {k: f"{len(df)} obs" for k, df in datos.items()})

  [caché] Coeficiente Gini
  [caché] NEET total (8.6.1)
  'onu_acceso_carretera' no en CATALOG
  [caché] Mortalidad por suicidio (3.4.2 ONU)
  1 obs (2005–2005)
  [fetch ONU SDG] Mujeres en dirección (5.5.2 ONU) (5.5.2)...
  Sin datos para México en 5.5.2
  [fetch ONU SDG] Emisiones GHG (13.2.2 ONU) (13.2.2)...
  Sin datos para México en 13.2.2
  'onu_ingreso_mediana' no en CATALOG
  guardado: data_cache/gini.csv
  guardado: data_cache/neet.csv
  guardado: data_cache/onu_suicidio.csv
Disponibles: {'gini': '10 obs', 'neet': '19 obs', 'onu_suicidio': '1 obs'}


In [ ]:
# %% ── Barrido de disponibilidad ONU SDG para México ─────────────────────────
import requests
import pandas as pd
import time

ONU_SDG_BASE  = "https://unstats.un.org/sdgs/UNSDGAPIV5/v1"
ONU_AREA_CODE = "484"  # México

def get_all_indicators():
    """Obtiene el catálogo completo de indicadores ODS de la API de ONU."""
    url = f"{ONU_SDG_BASE}/sdg/Indicator/List"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    return resp.json()

def check_indicator(code):
    """
    Checa disponibilidad de un indicador para México.
    Retorna dict con código, observaciones encontradas y años disponibles.
    """
    url = (
        f"{ONU_SDG_BASE}/sdg/Indicator/Data"
        f"?indicator={code}&areaCode={ONU_AREA_CODE}"
    )
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        raw   = resp.json()
        total = raw.get("totalElements", 0)
        years = sorted(set(
            int(obs["timePeriodStart"])
            for obs in raw.get("data", [])
            if "timePeriodStart" in obs
        ))
        return {
            "indicator": code,
            "total_obs": total,
            "n_years":   len(years),
            "year_min":  min(years) if years else None,
            "year_max":  max(years) if years else None,
            "status":    "completo" if len(years) >= 10
                    else "parcial"  if len(years) >  0
                    else "ausente",
        }
    except Exception as e:
        return {
            "indicator": code,
            "total_obs": 0,
            "n_years":   0,
            "year_min":  None,
            "year_max":  None,
            "status":    "error",
        }

# ── Paso 1: descargar catálogo ────────────────────────────────────────────────
print("Descargando catálogo de indicadores...")
catalogo = get_all_indicators()
print(f"Total indicadores en catálogo ONU: {len(catalogo)}")

# El catálogo tiene: code, description, goal, target
# Extraer solo los códigos
codigos = [ind["code"] for ind in catalogo]
print(f"Ejemplo de códigos: {codigos[:5]}")

Descargando catálogo de indicadores...
Total indicadores en catálogo ONU: 251
Ejemplo de códigos: ['1.1.1', '1.2.1', '1.2.2', '1.3.1', '1.4.1']


In [ ]:
# %% ── Barrido incremental ────────────────────────────────────────────────────
import json
from pathlib import Path

BARRIDO_CACHE = Path("data_cache/barrido_onu.json")

# Cargar progreso previo si existe
if BARRIDO_CACHE.exists():
    resultados = json.loads(BARRIDO_CACHE.read_text())
    ya_vistos  = {r["indicator"] for r in resultados}
    print(f"Retomando — ya procesados: {len(ya_vistos)}/{len(codigos)}")
else:
    resultados = []
    ya_vistos  = set()
    print(f"Iniciando barrido de {len(codigos)} indicadores...")

pendientes = [c for c in codigos if c not in ya_vistos]
print(f"Pendientes: {len(pendientes)}")

for i, code in enumerate(pendientes):
    resultado = check_indicator(code)
    resultados.append(resultado)

    # Guardar cada 10 para no perder progreso
    if (i + 1) % 10 == 0:
        BARRIDO_CACHE.write_text(json.dumps(resultados, ensure_ascii=False))
        completados = len(ya_vistos) + i + 1
        print(f"  {completados}/{len(codigos)} — último: {code} [{resultado['status']}]")

    time.sleep(0.3)  # cortesía a la API

# Guardar final
BARRIDO_CACHE.write_text(json.dumps(resultados, ensure_ascii=False))
print(f"\nBarrido completo. {len(resultados)} indicadores procesados.")

Iniciando barrido de 251 indicadores...
Pendientes: 251
  10/251 — último: 1.5.4 [completo]
  20/251 — último: 2.3.1 [ausente]
  30/251 — último: 3.1.2 [completo]
  40/251 — último: 3.5.2 [completo]
  50/251 — último: 3.a.1 [parcial]
  60/251 — último: 4.2.2 [parcial]
  70/251 — último: 5.2.1 [parcial]
  80/251 — último: 5.a.2 [parcial]
  90/251 — último: 6.5.2 [parcial]
  100/251 — último: 8.2.1 [completo]
  110/251 — último: 8.8.2 [completo]
  120/251 — último: 9.2.2 [completo]
  130/251 — último: 10.2.1 [completo]
  140/251 — último: 10.a.1 [parcial]
  150/251 — último: 11.5.3 [completo]
  160/251 — último: 12.2.1 [ausente]
  170/251 — último: 12.b.1 [completo]
  180/251 — último: 14.1.1 [completo]
  190/251 — último: 15.1.1 [parcial]
  200/251 — último: 15.9.1 [parcial]
  210/251 — último: 16.2.3 [parcial]
  220/251 — último: 16.7.1 [parcial]
  230/251 — último: 17.2.1 [ausente]
  240/251 — último: 17.11.1 [completo]
  250/251 — último: 17.19.1 [completo]

Barrido completo. 251 ind

In [ ]:
# %% ── Continuar el barrido ────────────────────────────────────────────────────
df_barrido = pd.DataFrame(resultados)

# Agregar metadata del catálogo (goal, descripción)
meta = {ind["code"]: ind for ind in catalogo}
df_barrido["goal"]        = df_barrido["indicator"].map(lambda c: meta[c]["goal"] if c in meta else None)
df_barrido["description"] = df_barrido["indicator"].map(lambda c: meta[c]["description"][:60] if c in meta else "")

print(df_barrido["status"].value_counts())
print()
print(df_barrido.groupby(["goal", "status"]).size().unstack(fill_value=0))

status
parcial     121
completo    105
ausente      24
error         1
Name: count, dtype: int64

status  ausente  completo  error  parcial
goal                                     
1             0         8      0        5
10            1         7      0        6
11            2         6      0        8
12            2         4      0        7
13            3         3      0        2
14            2         2      0        6
15            0         7      0        7
16            2         5      0       17
17            3        14      0        7
2             1        10      0        4
3             2        11      0       15
4             0         3      1        8
5             3         2      0        9
6             0         3      0        8
7             1         2      0        3
8             1         8      0        8
9             1        10      0        1


In [ ]:
# %% ── Heatmap de disponibilidad ONU SDG — México ────────────────────────────
import plotly.graph_objects as go
import numpy as np

# ── Preparar matriz ───────────────────────────────────────────────────────────

STATUS_VAL = {"completo": 2, "parcial": 1, "ausente": 0, "error": -1}
STATUS_LABEL = {2: "completo (≥10 años)", 1: "parcial (<10 años)", 0: "ausente", -1: "error"}

df_barrido["goal_num"] = pd.to_numeric(df_barrido["goal"], errors="coerce")
df_barrido["status_val"] = df_barrido["status"].map(STATUS_VAL)

# Número de posición del indicador dentro de su ODS
df_barrido["idx_within_goal"] = (
    df_barrido.sort_values("indicator")
              .groupby("goal_num")
              .cumcount()
)

goals     = sorted(df_barrido["goal_num"].dropna().unique())
max_cols  = df_barrido["idx_within_goal"].max() + 1

# Matriz (17 filas × max_cols columnas) — NaN donde no hay indicador
matrix     = np.full((len(goals), max_cols), np.nan)
text_matrix = np.full((len(goals), max_cols), "", dtype=object)

for _, row in df_barrido.dropna(subset=["goal_num"]).iterrows():
    g   = int(row["goal_num"]) - 1
    col = int(row["idx_within_goal"])
    matrix[g, col]      = row["status_val"]
    text_matrix[g, col] = (
        f"<b>{row['indicator']}</b><br>"
        f"{row['description'][:50]}<br>"
        f"Estado: {row['status']}<br>"
        f"Años con datos: {row['n_years']}"
        + (f" ({row['year_min']}–{row['year_max']})" if row['n_years'] > 0 else "")
    )

# ── Figura ────────────────────────────────────────────────────────────────────

fig = go.Figure(go.Heatmap(
    z=matrix,
    text=text_matrix,
    hovertemplate="%{text}<extra></extra>",
    colorscale=[
        [0.00, "#F1EFE8"],  # NaN → gris claro (celdas vacías)
        [0.00, "#F1EFE8"],
        [0.25, "#444441"],  # error → gris oscuro
        [0.50, "#E24B4A"],  # ausente → rojo
        [0.75, "#EF9F27"],  # parcial → amber
        [1.00, "#1D9E75"],  # completo → teal
    ],
    zmin=-1, zmax=2,
    showscale=False,
    xgap=2,
    ygap=2,
))

# Anotaciones de leyenda manual
for val, label, color in [
    (2,  "completo (≥10 años)", "#1D9E75"),
    (1,  "parcial (<10 años)",  "#EF9F27"),
    (0,  "ausente",             "#E24B4A"),
    (-1, "error",               "#888780"),
]:
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=10, color=color, symbol="square"),
        name=label,
        showlegend=True,
    ))

fig.update_layout(
    title="Disponibilidad de indicadores ODS — México (ONU SDG API)",
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(len(goals))),
        ticktext=[f"ODS {int(g)}" for g in goals],
        autorange="reversed",
    ),
    xaxis=dict(
        title="Indicadores dentro de cada ODS",
        showticklabels=False,
    ),
    height=600,
    plot_bgcolor="white",
    legend=dict(orientation="h", y=-0.08),
)

fig.show()

In [ ]:
# %% ── Diagnóstico: estructura real de respuesta ─────────────────────
import requests

url = (
    "https://unstats.un.org/sdgs/UNSDGAPIV5/v1/sdg/Indicator/Data"
    "?indicator=8.6.1&areaCode=484&timePeriodStart=2005&timePeriodEnd=2023"
)

resp = requests.get(url, timeout=15)
raw  = resp.json()

print("totalElements:", raw.get("totalElements"))
print("keys del primer obs:", list(raw["data"][0].keys()))
print("primer obs completo:")
import json
print(json.dumps(raw["data"][0], indent=2))

totalElements: 3
keys del primer obs: ['goal', 'target', 'indicator', 'series', 'seriesDescription', 'seriesCount', 'geoAreaCode', 'geoAreaName', 'timePeriodStart', 'value', 'valueType', 'time_detail', 'timeCoverage', 'upperBound', 'lowerBound', 'basePeriod', 'source', 'geoInfoUrl', 'footnotes', 'attributes', 'dimensions']
primer obs completo:
{
  "goal": [
    "8"
  ],
  "target": [
    "8.6"
  ],
  "indicator": [
    "8.6.1"
  ],
  "series": "SL_TLF_NEET",
  "seriesDescription": "Proportion of youth not in education, employment or training, by sex and age - 13th ICLS (%)",
  "seriesCount": "8917",
  "geoAreaCode": "484",
  "geoAreaName": "Mexico",
  "timePeriodStart": 2005,
  "value": "33.41",
  "valueType": "Float",
  "time_detail": null,
  "timeCoverage": null,
  "upperBound": null,
  "lowerBound": null,
  "basePeriod": null,
  "source": "LFS - National Occupation and Employment Survey",
  "geoInfoUrl": null,
  "footnotes": [
    "Repository: ILO-STATISTICS - Micro data processing 

In [ ]:
# %% ── Catálogo World Bank ────────────────────────────────────────────────────
import wbgapi as wb
import pandas as pd

def get_wb_sdg_indicators():
    """
    Obtiene indicadores del World Bank que corresponden a ODS.
    El WB tiene un topic específico para SDGs (topic id = 17).
    """
    indicators = []
    for ind in wb.series.list(topic=17):
        indicators.append({
            "code":        ind["id"],
            "description": ind["value"],
        })
    print(f"Total indicadores WB con topic SDG: {len(indicators)}")
    return indicators

catalogo_wb = get_wb_sdg_indicators()
print("Ejemplo:", catalogo_wb[:3])

In [ ]:
# %% ── Visualización (Plotly) ─────────────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_overlay(datasets: dict, title: str = "") -> go.Figure:
    """
    Indicadores superpuestos con eje Y por indicador.
    Cada serie es toggleable desde la leyenda.
    """
    fig = go.Figure()

    # Paleta fija por fuente
    palette = {
        "worldbank": "#1D9E75",
        "inegi":     "#534AB7",
        "alt1":      "#D85A30",
        "alt2":      "#BA7517",
    }
    palette_list = list(palette.values())

    # Un eje Y por indicador — así las distintas unidades no se pisan
    n = len(datasets)
    for i, (key, df) in enumerate(datasets.items()):
        color  = palette.get(df["source"].iloc[0], palette_list[i % len(palette_list)])
        label  = df["label"].iloc[0]
        unit   = df["unit"].iloc[0]
        yaxis  = "y" if i == 0 else f"y{i + 1}"

        fig.add_trace(go.Scatter(
            x=df["year"],
            y=df["value"],
            name=f"{label} [{unit}]",
            mode="lines+markers",
            marker=dict(size=5),
            line=dict(color=color, width=2),
            yaxis=yaxis,
        ))

    # Construir layout con ejes Y dinámicos
    layout = dict(
        title=dict(text=title, font=dict(size=14)),
        xaxis=dict(title="Año", showgrid=True, gridcolor="#eeeeee"),
        legend=dict(
            orientation="h",
            yanchor="bottom", y=-0.25,
            xanchor="left",   x=0,
        ),
        hovermode="x unified",
        plot_bgcolor="white",
        height=480,
    )

    # Posicionar ejes Y adicionales a la derecha
    offsets = [0, 0.08, 0.16, 0.24]
    for i in range(n):
        axis_key = "yaxis" if i == 0 else f"yaxis{i + 1}"
        color    = palette_list[i % len(palette_list)]
        layout[axis_key] = dict(
            title=list(datasets.values())[i]["unit"].iloc[0],
            title_font=dict(color=color),
            tickfont=dict(color=color),
            showgrid=(i == 0),
            gridcolor="#eeeeee",
            side="left" if i == 0 else "right",
            overlaying=None if i == 0 else "y",
            position=1 - offsets[min(i, len(offsets) - 1)],
        )

    fig.update_layout(**layout)
    return fig


def plot_grid(datasets: dict, cols: int = 2, title: str = "") -> go.Figure:
    """
    Cada indicador en su propio subplot.
    Útil para exploración inicial sin preocuparse por escala.
    """
    keys  = list(datasets.keys())
    n     = len(keys)
    rows  = (n + cols - 1) // cols
    palette_list = list({"worldbank": "#1D9E75", "inegi": "#534AB7",
                          "alt1": "#D85A30", "alt2": "#BA7517"}.values())

    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[datasets[k]["label"].iloc[0] for k in keys],
        vertical_spacing=0.12,
    )

    for i, key in enumerate(keys):
        df    = datasets[key]
        row   = i // cols + 1
        col   = i % cols + 1
        color = palette_list[i % len(palette_list)]

        fig.add_trace(
            go.Scatter(
                x=df["year"], y=df["value"],
                mode="lines+markers",
                marker=dict(size=4),
                line=dict(color=color, width=2),
                name=df["label"].iloc[0],
                showlegend=False,
            ),
            row=row, col=col,
        )
        # Etiqueta de unidad en eje Y de cada subplot
        fig.update_yaxes(title_text=df["unit"].iloc[0], row=row, col=col)

    fig.update_layout(
        title=dict(text=title, font=dict(size=14)),
        height=300 * rows,
        plot_bgcolor="white",
        paper_bgcolor="white",
    )
    fig.update_xaxes(showgrid=True, gridcolor="#eeeeee")
    fig.update_yaxes(showgrid=True, gridcolor="#eeeeee")
    return fig




In [ ]:
# %% ── NEET apilado (M+H) vs Gini ────────────────────────────────────────────

df_f    = datos["neet_f"]
df_m    = datos["neet_m"]
df_gini = datos["gini"]

fig = go.Figure()

# Barras apiladas — mujeres abajo, hombres arriba
fig.add_trace(go.Bar(
    x=df_f["year"], y=df_f["value"],
    name="NEET mujeres",
    marker_color="#D4537E",
    opacity=0.85,
    yaxis="y1",
))
fig.add_trace(go.Bar(
    x=df_m["year"], y=df_m["value"],
    name="NEET hombres",
    marker_color="#378ADD",
    opacity=0.85,
    yaxis="y1",
))

# Línea Gini en eje secundario
fig.add_trace(go.Scatter(
    x=df_gini["year"], y=df_gini["value"],
    name="Gini (desigualdad)",
    mode="lines+markers",
    marker=dict(size=5),
    line=dict(color="#1D9E75", width=2.5),
    yaxis="y2",
))

fig.update_layout(
    barmode="stack",
    title="México: NEET por género vs Desigualdad (Gini) 2005–2023",
    xaxis=dict(title="Año", showgrid=False),
    yaxis=dict(
        title="% jóvenes 15-24 NEET",
        showgrid=True,
        gridcolor="#eeeeee",
    ),
    yaxis2=dict(
        title="Gini",
        overlaying="y",
        side="right",
        showgrid=False,
        title_font=dict(color="#1D9E75"),
        tickfont=dict(color="#1D9E75"),
    ),
    legend=dict(orientation="h", y=-0.2),
    plot_bgcolor="white",
    hovermode="x unified",
    height=480,
)

fig.show()



In [ ]:
# %% ── Gráficas ───────────────────────────────────────────────────────────────

# Overlay: NEET total vs Gini
plot_overlay(
    {k: datos[k] for k in ["neet", "gini"] if k in datos},
    title="México: NEET vs Desigualdad (Gini) 2005–2023",
)



In [ ]:
# Overlay: NEET total vs Gini
plot_overlay(
    {k: datos[k] for k in ["neet", "gini"] if k in datos},
    title="México: NEET vs Desigualdad (Gini) 2005–2023",
)

# Overlay: brecha de género en NEET
plot_overlay(
    {k: datos[k] for k in ["neet_f", "neet_m"] if k in datos},
    title="México: NEET mujeres vs hombres 2005–2023",
)

# Grid: todos los indicadores disponibles
plot_grid(
    datos,
    title="México: Indicadores ODS",
)